# Notebook 1: Data Collection & Historical Database

**Goal:** Build the foundation dataset from Tank01 API

**Output:** `yakos_historical_master.parquet` — one row per player per game date

**Schema:** `player_id, player_name, team, pos, game_date, minutes, dk_fp_actual, salary, tank01_proj, rg_proj, rg_ownership, rg_minutes, vegas_total, vegas_spread`

## Setup
Run in Google Colab or locally. Set `RAPIDAPI_KEY` before running.

In [ ]:
# Install dependencies (Colab)
# !pip install requests pandas pyarrow -q

import os
import time
import requests
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path

# --- Configuration ---
RAPIDAPI_KEY = os.environ.get('RAPIDAPI_KEY', '')  # Set via env or paste here
TANK01_HOST  = 'tank01-fantasy-stats.p.rapidapi.com'
OUTPUT_PATH  = 'yakos_historical_master.parquet'

# Date range to collect (last 60 days)
END_DATE   = datetime.today()
START_DATE = END_DATE - timedelta(days=60)

def headers():
    return {'x-rapidapi-key': RAPIDAPI_KEY, 'x-rapidapi-host': TANK01_HOST}

print('Config ready. RAPIDAPI_KEY set:', bool(RAPIDAPI_KEY))

## Step 1: Fetch Season Averages (All Active Players)

Uses `getNBATeams` with `rosters=true` and `statsToGet=averages` to pull per-player season averages: pts, reb, ast, stl, blk, tov, min, fg%, 3pm, etc.

In [ ]:
def fetch_season_averages():
    """Fetch season averages for all active NBA players via getNBATeams."""
    url = f'https://{TANK01_HOST}/getNBATeams'
    params = {'rosters': 'true', 'statsToGet': 'averages'}
    resp = requests.get(url, headers=headers(), params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    body = data.get('body', data) if isinstance(data, dict) else data

    rows = []
    teams = body if isinstance(body, list) else body.get('teams', body.get('body', []))
    for team in teams:
        if not isinstance(team, dict):
            continue
        team_abv = team.get('teamAbv', '')
        roster = team.get('roster', team.get('rosterPlayers', []))
        if not isinstance(roster, list):
            continue
        for player in roster:
            if not isinstance(player, dict):
                continue
            stats = player.get('stats', player.get('averages', {}))
            rows.append({
                'player_id':   player.get('playerID', ''),
                'player_name': player.get('longName', player.get('playerName', '')),
                'team':        team_abv,
                'pos':         player.get('pos', ''),
                'avg_pts':     float(stats.get('pts', 0) or 0),
                'avg_reb':     float(stats.get('reb', 0) or 0),
                'avg_ast':     float(stats.get('ast', 0) or 0),
                'avg_stl':     float(stats.get('stl', 0) or 0),
                'avg_blk':     float(stats.get('blk', 0) or 0),
                'avg_tov':     float(stats.get('TOV', stats.get('tov', 0)) or 0),
                'avg_min':     float(stats.get('min', stats.get('mins', 0)) or 0),
                'avg_3pm':     float(stats.get('tpm', stats.get('fg3m', 0)) or 0),
                'fg_pct':      float(stats.get('fgPct', stats.get('fg_pct', 0)) or 0),
            })

    df = pd.DataFrame(rows)
    print(f'Season averages: {len(df)} players from {df["team"].nunique()} teams')
    return df


season_avgs = fetch_season_averages()
season_avgs.head()

## Step 2: Fetch DFS Slates (Salary + Tank01 Projections)

Uses `getNBADFS` for each historical date to capture salary, `fantasyPoints` projection, and ownership projections.

In [ ]:
def fetch_dfs_slate(date_str):
    """Fetch DK DFS slate from Tank01 for a given date (YYYYMMDD)."""
    url = f'https://{TANK01_HOST}/getNBADFS'
    resp = requests.get(url, headers=headers(), params={'date': date_str}, timeout=30)
    if resp.status_code != 200:
        return pd.DataFrame()
    data = resp.json()
    body = data.get('body', data) if isinstance(data, dict) else data

    # Unwrap DraftKings key if nested
    if isinstance(body, dict):
        for key in ('DraftKings', 'DK', 'dk', 'draftkings', 'players'):
            if key in body and isinstance(body[key], list):
                body = body[key]
                break

    rows = []
    for p in (body if isinstance(body, list) else []):
        if not isinstance(p, dict):
            continue
        rows.append({
            'player_id':     str(p.get('playerID', '')),
            'player_name':   p.get('longName', p.get('player', '')),
            'team':          p.get('teamAbv', p.get('team', '')).upper(),
            'pos':           p.get('pos', ''),
            'game_date':     f'{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}',
            'salary':        int(float(p.get('salary', 0) or 0)),
            'tank01_proj':   float(p.get('fantasyPoints', p.get('fpts', 0)) or 0),
            'rg_proj':       None,
            'rg_ownership':  None,
            'rg_minutes':    None,
        })

    return pd.DataFrame(rows)


def collect_dfs_slates(start_date, end_date):
    """Collect DFS slates for a date range."""
    all_slates = []
    current = start_date
    while current <= end_date:
        date_str = current.strftime('%Y%m%d')
        df = fetch_dfs_slate(date_str)
        if len(df) > 0:
            all_slates.append(df)
            print(f'  {date_str}: {len(df)} players')
        current += timedelta(days=1)
        time.sleep(0.3)  # Rate limit

    if not all_slates:
        return pd.DataFrame()
    return pd.concat(all_slates, ignore_index=True)


print('Collecting DFS slates...')
dfs_slates = collect_dfs_slates(START_DATE, END_DATE)
print(f'Total: {len(dfs_slates)} rows across {dfs_slates["game_date"].nunique()} dates')

## Step 3: Fetch Actual DK FP from Box Scores

Uses `getNBAGamesForDate` + `getNBABoxScore` to get real game stats and compute actual DK fantasy points.

In [ ]:
def calc_dk_fp(p):
    """Compute DraftKings NBA FP from raw box score stats."""
    def f(*keys):
        for k in keys:
            v = p.get(k)
            if v is not None:
                try: return float(v)
                except: pass
        return 0.0

    pts  = f('pts', 'PTS', 'points')
    reb  = f('reb', 'REB', 'rebounds', 'totalReb')
    ast  = f('ast', 'AST', 'assists')
    stl  = f('stl', 'STL', 'steals')
    blk  = f('blk', 'BLK', 'blocks')
    tov  = f('TOV', 'tov', 'turnovers')
    fg3m = f('tpm', 'fg3m', 'threePM', '3PM')
    fp   = pts + fg3m*0.5 + reb*1.25 + ast*1.5 + stl*2 + blk*2 - tov*0.5
    cats = sum(1 for c in [pts, reb, ast, stl, blk] if c >= 10)
    if cats >= 3: fp += 3.0
    elif cats >= 2: fp += 1.5
    return round(fp, 2)


def fetch_box_score_actuals(date_str):
    """Fetch actual DK FP for all players on a given date."""
    formatted = f'{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}'
    # Get game IDs
    resp = requests.get(
        f'https://{TANK01_HOST}/getNBAGamesForDate',
        headers=headers(), params={'gameDate': formatted}, timeout=30
    )
    if resp.status_code != 200:
        return pd.DataFrame()
    body = resp.json().get('body', {})
    games = body.get('games', []) if isinstance(body, dict) else (body if isinstance(body, list) else [])

    rows = []
    for game in games:
        game_id = game.get('gameID', game.get('gameId', ''))
        if not game_id:
            continue
        box_resp = requests.get(
            f'https://{TANK01_HOST}/getNBABoxScore',
            headers=headers(), params={'gameID': game_id}, timeout=30
        )
        if box_resp.status_code != 200:
            continue
        box = box_resp.json().get('body', {})
        players = box.get('playerStats', []) if isinstance(box, dict) else []
        for p in players:
            if not isinstance(p, dict):
                continue
            name = p.get('displayName') or p.get('longName') or p.get('playerName', '')
            if not name:
                continue
            # Try pre-computed DK FP first
            fp_val = p.get('fantasyPoints')
            if isinstance(fp_val, dict):
                fp_val = fp_val.get('DraftKings') or fp_val.get('dkPoints')
            try:
                fp = float(fp_val) if fp_val is not None else calc_dk_fp(p)
            except:
                fp = calc_dk_fp(p)
            mins_raw = p.get('min', p.get('mins', p.get('minutes', 0)))
            try:
                minutes = float(str(mins_raw).split(':')[0]) if mins_raw else 0.0
            except:
                minutes = 0.0
            rows.append({
                'player_name': name,
                'game_date':   f'{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}',
                'dk_fp_actual': fp,
                'minutes':     minutes,
            })
        time.sleep(0.2)

    return pd.DataFrame(rows)


# Collect actuals for all dates in our DFS slate range
all_actuals = []
for date_val in dfs_slates['game_date'].unique():
    date_str = date_val.replace('-', '')
    df_act = fetch_box_score_actuals(date_str)
    if len(df_act) > 0:
        all_actuals.append(df_act)
        print(f'  {date_val}: {len(df_act)} player actuals')

actuals_df = pd.concat(all_actuals, ignore_index=True) if all_actuals else pd.DataFrame()
print(f'Total actuals: {len(actuals_df)} rows')

## Step 4: Merge RotoGrinders CSVs (If Available)

Load any saved RG exports to capture FPTS, OWNERSHIP, MINUTES, FLOOR, CEIL columns.

In [ ]:
def load_rg_exports(rg_dir='.'):
    """Load all RotoGrinders CSV exports from a directory."""
    rg_col_map = {
        'Name':      'player_name',
        'FPTS':      'rg_proj',
        'Ownership': 'rg_ownership',
        'Minutes':   'rg_minutes',
        'Floor':     'rg_floor',
        'Ceiling':   'rg_ceil',
        'Salary':    'salary',
    }
    from pathlib import Path
    all_rg = []
    for csv_path in Path(rg_dir).glob('NBADK*.csv'):
        df = pd.read_csv(csv_path)
        df = df.rename(columns={k: v for k, v in rg_col_map.items() if k in df.columns})
        # Infer date from filename e.g. NBADK20260227.csv
        stem = csv_path.stem
        digits = ''.join(c for c in stem if c.isdigit())
        if len(digits) >= 8:
            d = digits[:8]
            df['game_date'] = f'{d[:4]}-{d[4:6]}-{d[6:]}'
        all_rg.append(df)
        print(f'  Loaded {csv_path.name}: {len(df)} rows')

    if not all_rg:
        return pd.DataFrame()
    return pd.concat(all_rg, ignore_index=True)


# Change rg_dir to wherever your RG CSVs are stored
rg_df = load_rg_exports(rg_dir='.')
print(f'RG data: {len(rg_df)} rows')

## Step 5: Fetch Betting Odds

Uses `getNBABettingOdds` for game totals and spreads — critical pace/environment signals.

In [ ]:
def fetch_betting_odds(date_str):
    """Fetch game totals and spreads for a date."""
    url = f'https://{TANK01_HOST}/getNBABettingOdds'
    resp = requests.get(url, headers=headers(), params={'gameDate': date_str}, timeout=30)
    if resp.status_code != 200:
        return pd.DataFrame()
    data = resp.json()
    body = data.get('body', data)
    games = body if isinstance(body, list) else body.get('games', body.get('odds', []))

    rows = []
    for game in (games if isinstance(games, list) else []):
        if not isinstance(game, dict):
            continue
        home = game.get('homeTeam', game.get('home', ''))
        away = game.get('awayTeam', game.get('away', ''))
        total_raw = game.get('total', game.get('overUnder', game.get('o/u', None)))
        spread_raw = game.get('homeSpread', game.get('spread', None))
        try:
            vegas_total = float(total_raw) if total_raw else None
        except: vegas_total = None
        try:
            vegas_spread = float(spread_raw) if spread_raw else None
        except: vegas_spread = None
        game_date = f'{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}'
        for team in [home, away]:
            if team:
                rows.append({
                    'team': str(team).upper(),
                    'game_date': game_date,
                    'vegas_total': vegas_total,
                    'vegas_spread': vegas_spread if team == home else (
                        -vegas_spread if vegas_spread else None
                    ),
                })

    return pd.DataFrame(rows)


# Collect odds for all dates
all_odds = []
for date_val in dfs_slates['game_date'].unique():
    date_str = date_val.replace('-', '')
    df_odds = fetch_betting_odds(date_str)
    if len(df_odds) > 0:
        all_odds.append(df_odds)

odds_df = pd.concat(all_odds, ignore_index=True) if all_odds else pd.DataFrame(
    columns=['team', 'game_date', 'vegas_total', 'vegas_spread']
)
print(f'Odds data: {len(odds_df)} rows')

## Step 6: Build Master Parquet

Merge everything into `yakos_historical_master.parquet`.

In [ ]:
def build_master_parquet(dfs_slates, actuals_df, rg_df, odds_df, season_avgs):
    """Merge all data sources into the master historical parquet."""
    master = dfs_slates.copy()

    # Merge actuals
    if len(actuals_df) > 0:
        master = master.merge(
            actuals_df[['player_name', 'game_date', 'dk_fp_actual', 'minutes']],
            on=['player_name', 'game_date'], how='left'
        )
    else:
        master['dk_fp_actual'] = None
        master['minutes'] = None

    # Merge RG data
    if len(rg_df) > 0 and 'player_name' in rg_df.columns and 'game_date' in rg_df.columns:
        rg_cols = ['player_name', 'game_date']
        for c in ['rg_proj', 'rg_ownership', 'rg_minutes']:
            if c in rg_df.columns:
                rg_cols.append(c)
        master = master.merge(rg_df[rg_cols], on=['player_name', 'game_date'], how='left')

    # Merge betting odds
    if len(odds_df) > 0 and 'team' in odds_df.columns:
        master = master.merge(
            odds_df[['team', 'game_date', 'vegas_total', 'vegas_spread']],
            on=['team', 'game_date'], how='left'
        )
    else:
        master['vegas_total']  = None
        master['vegas_spread'] = None

    # Merge season averages (season-level stats, not per-game)
    if len(season_avgs) > 0:
        sa_cols = ['player_id', 'avg_pts', 'avg_reb', 'avg_ast', 'avg_min', 'avg_3pm']
        sa_cols = [c for c in sa_cols if c in season_avgs.columns]
        master = master.merge(season_avgs[sa_cols], on='player_id', how='left')

    # Enforce schema column order
    schema_cols = [
        'player_id', 'player_name', 'team', 'pos', 'game_date',
        'minutes', 'dk_fp_actual', 'salary', 'tank01_proj',
        'rg_proj', 'rg_ownership', 'rg_minutes',
        'vegas_total', 'vegas_spread',
    ]
    for c in schema_cols:
        if c not in master.columns:
            master[c] = None

    master = master[schema_cols + [c for c in master.columns if c not in schema_cols]]
    master = master.drop_duplicates(subset=['player_name', 'game_date'])
    return master


master_df = build_master_parquet(dfs_slates, actuals_df, rg_df, odds_df, season_avgs)
master_df.to_parquet(OUTPUT_PATH, index=False)

print(f'\n=== Master Parquet Saved ===')
print(f'Rows: {len(master_df)}')
print(f'Dates: {master_df["game_date"].nunique()}')
print(f'Players: {master_df["player_name"].nunique()}')
print(f'Columns: {list(master_df.columns)}')
master_df.head(10)